In [1]:
# Library imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
from scipy.stats import wilcoxon
from scipy import stats

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display the dataframe to fit nicely on the screen
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [2]:
# Path to the experiments directory
dir_experiments = "../experiments/nsuperpixels_filterReduction/results"

# Dicionários separados
results_files = defaultdict(dict)
superpixel_images = defaultdict(dict)

# Traverse the experiments directory
for superpixel_folder in sorted(os.listdir(dir_experiments)):
    superpixel_path = os.path.join(dir_experiments, superpixel_folder)
    if os.path.isdir(superpixel_path) and superpixel_folder.startswith("super"):
        # Armazena arquivos de resultados
        for file in sorted(os.listdir(superpixel_path)):
            if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[superpixel_folder][file] = os.path.join(superpixel_path, file)
        # Armazena imagens dos superpixels
        for seed_folder in sorted(os.listdir(superpixel_path)):
            seed_path = os.path.join(superpixel_path, seed_folder)
            if os.path.isdir(seed_path) and seed_folder.startswith("superpixels_seed"):
                superpixel_images[superpixel_folder][seed_folder] = []
                for img_file in sorted(os.listdir(seed_path)):
                    superpixel_images[superpixel_folder][seed_folder].append(os.path.join(seed_path, img_file))
                    
# Create a structured DataFrame to store the results
results_data = []
superpixels_values = []

for superpixel, contents in results_files.items():
    # print(f"{superpixel}: {len(contents)} files")
    superpixels_values.append(int(superpixel.replace('super', '').replace('_filterReduction', '')))
    for seed, results_file in contents.items():
        if isinstance(results_file, str) and results_file.endswith('_results.csv'):
            seed_value = int(seed.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    class_accuracies = list(map(float, lines[0].strip().split(';')[:9]))
                    class1_accuracy, class2_accuracy = class_accuracies[0], class_accuracies[1]
                    class3_accuracy, class4_accuracy = class_accuracies[2], class_accuracies[3]
                    class5_accuracy, class6_accuracy = class_accuracies[4], class_accuracies[5]
                    class7_accuracy, class8_accuracy = class_accuracies[6], class_accuracies[7]
                    class9_accuracy = class_accuracies[8]
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    results_data.append({
                        'superpixel': int(superpixel.replace('super', '').replace('_filterReduction', '')),
                        'filterReduction': 'filterReduction' in superpixel,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'class3_accuracy': class3_accuracy,
                        'class4_accuracy': class4_accuracy,
                        'class5_accuracy': class5_accuracy,
                        'class6_accuracy': class6_accuracy,
                        'class7_accuracy': class7_accuracy,
                        'class8_accuracy': class8_accuracy,
                        'class9_accuracy': class9_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })

results_data = sorted(results_data, key=lambda x: x['superpixel'])

# Convert the results data into a DataFrame
df_nsuper = pd.DataFrame(results_data) 

# Save the DataFrame to a CSV file
df_nsuper.to_csv('nsuperpixelsFilterReduction_results_summary.csv', index=False)

In [3]:
df_nsuper.head(100)

,superpixel,filterReduction,seed,class1_accuracy,class2_accuracy,class3_accuracy,class4_accuracy,class5_accuracy,class6_accuracy,class7_accuracy,class8_accuracy,class9_accuracy,kappa,global_accuracy,nfeat
0,50,False,1011,0.965517,0.950,0.837838,0.934426,0.958580,0.984043,0.852459,0.940678,0.958134,0.916062,0.953070,211600
1,50,False,1213,0.965517,0.975,0.851351,0.918033,0.970414,0.984043,0.918033,0.974576,0.965909,0.932772,0.962456,211600
2,50,False,123,0.954023,1.000,0.797297,0.950820,0.946746,0.973404,0.868852,0.949153,0.949761,0.903883,0.946030,211600
3,50,False,2735,0.948276,0.975,0.756757,0.918033,0.952663,0.989362,0.901639,0.949153,0.962321,0.917069,0.953852,211600
4,50,False,42,0.850575,1.000,0.756757,0.754098,0.863905,0.968085,0.803279,0.923729,0.892943,0.802751,0.887368,211600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,200,True,456,0.833333,0.875,0.621622,0.918033,0.721893,0.920213,0.770492,0.881356,0.762560,0.646103,0.783340,25392
76,200,True,6854,0.816092,0.800,0.716216,0.606557,0.751479,0.925532,0.819672,0.813559,0.821770,0.685258,0.815409,25392
77,200,True,7580,0.896552,0.850,0.689189,0.803279,0.792899,0.936170,0.639344,0.805085,0.755981,0.644952,0.781384,25392
78,200,True,789,0.867816,0.975,0.729730,0.754098,0.893491,0.962766,0.737705,0.838983,0.902512,0.805243,0.889714,25392


In [6]:
# Define the metrics to analyze
metrics = ['class1_accuracy', 'class2_accuracy', 'class3_accuracy', 'class4_accuracy', 'class5_accuracy', 'class6_accuracy', 'class7_accuracy', 'class8_accuracy', 'class9_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Group results by number of superpixels and calculate mean and standard deviation for each metric
summary_stats = df_nsuper.groupby(['superpixel', 'filterReduction'])[metrics[:]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# # Select only the metrics of interest for visualization and highlight the highest values
# summary_stats = summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class1_accuracy_std', 'class2_accuracy_mean', 'class2_accuracy_std', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'], color='gray'
# )

# # Format values as percentages for better presentation
# summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Mostrar apenas algumas colunas específicas
# summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean']]
summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class2_accuracy_mean', 'class3_accuracy_mean', 'class4_accuracy_mean', 'class5_accuracy_mean', 'class6_accuracy_mean', 'class7_accuracy_mean', 'class8_accuracy_mean', 'class9_accuracy_mean', 'kappa_mean', 'global_accuracy_mean']]

# latex_table = summary_stats.to_latex(float_format="%.6f")
# print(latex_table)

# summary_stats[['superpixel_', 'filterReduction_', 'kappa_mean', 'global_accuracy_mean']]

# # Renomeia as colunas para facilitar o acesso
# summary_stats_df.columns = ['superpixel', 'filterReduction'] + [f"{m}_{stat}" for m in metrics[:-1] for stat in ['mean', 'std']]

# # Ordena pelo número de superpixels e filterReduction
# summary_stats_df = summary_stats_df.sort_values(['superpixel', 'filterReduction']).reset_index(drop=True)

# summary_stats_df    

# # Destaca os maiores valores das métricas principais
# styled_summary = summary_stats_df.style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'],
#     color='gray'
# ).format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# styled_summary


# Display the styled DataFrame
# summary_stats

# Export the styled summary table to LaTeX code (without styling)
# latex_table = df_nsuper.groupby('superpixel')[metrics[:-1]].agg(['mean', 'std']).loc[superpixels_values].to_latex(float_format="%.4f")
# print(latex_table)

,superpixel_,filterReduction_,class1_accuracy_mean,class2_accuracy_mean,class3_accuracy_mean,class4_accuracy_mean,class5_accuracy_mean,class6_accuracy_mean,class7_accuracy_mean,class8_accuracy_mean,class9_accuracy_mean,kappa_mean,global_accuracy_mean
0,50,False,0.940230,0.9750,0.793243,0.916394,0.951479,0.982447,0.885246,0.952543,0.954904,0.908426,0.948651
1,50,True,0.887356,0.8750,0.740541,0.816393,0.785207,0.919149,0.762295,0.849152,0.778289,0.679906,0.800743
2,100,False,0.947701,0.9725,0.806757,0.929508,0.957396,0.982979,0.901639,0.966102,0.963278,0.922402,0.956746
3,100,True,0.899425,0.9050,0.722973,0.803279,0.806509,0.939894,0.796721,0.874576,0.792703,0.707814,0.815565
4,150,False,0.944253,0.9725,0.808108,0.934426,0.955621,0.982979,0.903279,0.967797,0.963277,0.922280,0.956668
5,150,True,0.879310,0.9000,0.754054,0.814754,0.819527,0.948936,0.770492,0.875424,0.853289,0.755811,0.855847
6,200,False,0.949425,0.9700,0.798648,0.921312,0.954438,0.981915,0.888524,0.966102,0.964175,0.921651,0.956394
7,200,True,0.877012,0.8850,0.740541,0.822951,0.800000,0.947872,0.744262,0.848305,0.756758,0.674755,0.788893


In [4]:
# Wilcoxon signed-rank test para todos os pares de superpixel apenas para filterReduction_ = True
wilcoxon_results = pd.DataFrame(columns=['superpixel1', 'superpixel2', 'statistic', 'p_value'])

# Seleciona apenas os superpixels com filterReduction = True e ordena
superpixels_true = sorted(df_nsuper[df_nsuper['filterReduction'] == True]['superpixel'].unique())

for i in range(len(superpixels_true)):
    for j in range(i + 1, len(superpixels_true)):
        sp1 = superpixels_true[i]
        sp2 = superpixels_true[j]
        
        data_sp1 = df_nsuper[(df_nsuper['superpixel'] == sp1) & (df_nsuper['filterReduction'] == True)]['global_accuracy']
        data_sp2 = df_nsuper[(df_nsuper['superpixel'] == sp2) & (df_nsuper['filterReduction'] == True)]['global_accuracy']
        
        # Só faz o teste se os tamanhos das amostras forem iguais e maiores que 0
        if len(data_sp1) > 0 and len(data_sp2) > 0 and len(data_sp1) == len(data_sp2):
            statistic, p_value = wilcoxon(data_sp1, data_sp2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'superpixel1': sp1,
                'superpixel2': sp2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Wilcoxon test results:")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparisons with statistically significant difference:")
print(significant)
print(f"Total significant comparisons: {len(significant)} out of {len(wilcoxon_results)}")


Wilcoxon test results:
   superpixel1  superpixel2  statistic   p_value
0           50          100       24.0  0.769531
1           50          150        3.0  0.009766
2           50          200       21.0  0.556641
3          100          150       17.0  0.322266
4          100          200       24.0  0.769531
5          150          200        4.0  0.013672
Comparisons with statistically significant difference:
   superpixel1  superpixel2  statistic   p_value
1           50          150        3.0  0.009766
5          150          200        4.0  0.013672
Total significant comparisons: 2 out of 6


In [5]:
# Wilcoxon signed-rank test para todos os pares de superpixel apenas para filterReduction_ = True
wilcoxon_results = pd.DataFrame(columns=['superpixel1', 'superpixel2', 'statistic', 'p_value'])

# Seleciona apenas os superpixels com filterReduction = True e ordena
superpixels_true = sorted(df_nsuper[df_nsuper['filterReduction'] == False]['superpixel'].unique())

for i in range(len(superpixels_true)):
    for j in range(i + 1, len(superpixels_true)):
        sp1 = superpixels_true[i]
        sp2 = superpixels_true[j]

        data_sp1 = df_nsuper[(df_nsuper['superpixel'] == sp1) & (df_nsuper['filterReduction'] == False)]['global_accuracy']
        data_sp2 = df_nsuper[(df_nsuper['superpixel'] == sp2) & (df_nsuper['filterReduction'] == False)]['global_accuracy']

        # Só faz o teste se os tamanhos das amostras forem iguais e maiores que 0
        if len(data_sp1) > 0 and len(data_sp2) > 0 and len(data_sp1) == len(data_sp2):
            statistic, p_value = wilcoxon(data_sp1, data_sp2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'superpixel1': sp1,
                'superpixel2': sp2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Wilcoxon test results:")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparisons with statistically significant difference:")
print(significant)
print(f"Total significant comparisons: {len(significant)} out of {len(wilcoxon_results)}")


Wilcoxon test results:
   superpixel1  superpixel2  statistic   p_value
0           50          100       18.0  0.375000
1           50          150       18.5  0.390625
2           50          200       16.0  0.476562
3          100          150       27.0  1.000000
4          100          200       22.0  1.000000
5          150          200       25.0  0.845703
Comparisons with statistically significant difference:
Empty DataFrame
Columns: [superpixel1, superpixel2, statistic, p_value]
Index: []
Total significant comparisons: 0 out of 6
